In [1]:
MODE = 'PDE'

In [2]:
import pandas as pd
import numpy as np
import os
import glob

if MODE == 'PDE':
    df = pd.read_csv('output/individual/catalogue/protagonists_raw.csv') # Load PDE protagonists catalogue

In [3]:
def get_player_metrics(df, target_player_id, window_sec=300, event_type=None, is_post_window=False):
        
    if df.empty or window_sec <= 0:
        return {m: 0.0 for m in ['pass_acc_ratio', 'fwd_pass_ratio', 'long_ball_ratio', 'xg_rate', 
                                'shot_rate', 'avg_xg', 'dribble_rate', 'turnover_rate', 
                                'foul_rate', 'press_intensity']}
    
    # Filter for the specific player
    player_df = df[df.player_id == target_player_id].copy()
    
    # Filter out non-open play events for passing and shooting metrics
    if 'pass_type' in player_df.columns:
        player_df = player_df[~player_df.pass_type.isin(['Corner', 'Free Kick', 'Goal Kick', 'Throw-in'])]
    if 'shot_type' in player_df.columns:
        player_df = player_df[~player_df.shot_type.isin(['Penalty', 'Free Kick'])]

    res = {}
    
    # Passing Metrics
    passes = player_df[player_df.type == 'Pass']
    n_passes = len(passes)
    if n_passes > 0:
        # Pass completion rate: Successful / Total
        res['pass_acc_ratio'] = len(passes[passes.pass_outcome.isnull()]) / n_passes
        
        # Forward pass ratio: Forward / Total
        res['fwd_pass_ratio'] = (passes['end_x'] > passes['x']).sum() / n_passes
        
        # Long ball ratio: Long (>32m) / Total
        distances = np.sqrt((passes['end_x'] - passes['x'])**2 + (passes['end_y'] - passes['y'])**2)
        res['long_ball_ratio'] = (distances > 32).sum() / n_passes
    else:
        res['pass_acc_ratio'] = res['fwd_pass_ratio'] = res['long_ball_ratio'] = 0.0

    # Shooting Metrics
    shots = player_df[player_df.type == 'Shot']
    if 'total_seconds' in shots.columns:
        shots = shots.sort_values('total_seconds')

    if is_post_window and event_type in {'Converted Turnover', 'Final Third Dribble'} and len(shots) > 0:
        shots = shots.iloc[1:]

    res['xg_rate'] = shots.shot_statsbomb_xg.sum() if len(shots) > 0 else 0.0
    res['shot_rate'] = len(shots)
    res['avg_xg'] = shots.shot_statsbomb_xg.mean() if len(shots) > 0 else 0.0

    # Dribbling and Turnover Metrics
    res['dribble_rate'] = len(player_df[player_df.type == 'Dribble'])
    res['turnover_rate'] = len(player_df[player_df.type.isin(['Dispossessed', 'Miscontrol'])])

    # Defensive Metrics
    res['foul_rate'] = len(player_df[player_df.type == 'Foul Committed'])
    res['press_intensity'] = len(player_df[player_df.type == 'Pressure'])
    
    return res

In [4]:
def get_player_match_baseline_stats(events, target_player_id, window_sec=300):

    if events.empty:
        return {}

    # End of match in seconds
    match_length_seconds = events['total_seconds'].max()

    # Determine the time window during which the player was on the pitch
    start_time = 0.0
    end_time = match_length_seconds
    
    sub_col = 'substitution_replacement_id'
    
    # Substitution check: if the player was substituted on, adjust start_time
    if sub_col in events.columns:
        sub_on = events[(events['type'] == 'Substitution') & (events[sub_col] == target_player_id)]
        if not sub_on.empty:
            start_time = sub_on.iloc[0]['total_seconds']
            
    # Substitution check: if the player was substituted off, adjust end_time
    sub_off = events[(events['type'] == 'Substitution') & (events['player_id'] == target_player_id)]
    if not sub_off.empty:
        end_time = min(end_time, sub_off.iloc[0]['total_seconds'])
        
    # Red card check: if the player received a red card, adjust end_time
    if 'foul_committed_card' in events.columns:
        red_foul = events[(events['player_id'] == target_player_id) & 
                          (events['foul_committed_card'].isin(['Red Card', 'Second Yellow']))]
        if not red_foul.empty:
            end_time = min(end_time, red_foul.iloc[0]['total_seconds'])
            
    if 'bad_behaviour_card' in events.columns:
        red_bb = events[(events['player_id'] == target_player_id) & 
                        (events['bad_behaviour_card'].isin(['Red Card', 'Second Yellow']))]
        if not red_bb.empty:
            end_time = min(end_time, red_bb.iloc[0]['total_seconds'])

    # Create time windows for the player's presence on the pitch
    match_windows_metrics = []
    
    for start_t in np.arange(start_time, end_time, window_sec):
        end_t = start_t + window_sec

        # If the end of the window exceeds the player's time on the pitch, break the loop (Only full 5 minute windows are considered)
        if end_t > end_time:
            break

        # Filter events for the current time window
        window_events = events[(events['total_seconds'] >= start_t) & 
                               (events['total_seconds'] < end_t)]
        
        # Get player metrics for the current window
        metrics = get_player_metrics(window_events, target_player_id, window_sec=window_sec)
        match_windows_metrics.append(metrics)

    # Baseline statistics: mean and standard deviation for each metric across all windows
    if len(match_windows_metrics) < 2:
        # If there are fewer than 2 windows, we cannot compute standard deviation, so we return an empty dictionary
        return {}
        
    metrics_df = pd.DataFrame(match_windows_metrics)
    
    baseline_stats = {}
    for col in metrics_df.columns:
        baseline_stats[f'mean_{col}'] = metrics_df[col].mean()
        baseline_stats[f'std_{col}'] = metrics_df[col].std()
        
    return baseline_stats

In [5]:
def calculate_individual_behavioral_response(event_row, events, home_team_name, W_seconds=300):
    
    t = event_row['total_seconds']
    target_player_id = event_row['player_id']
    team = event_row['affected_team']
    period = event_row['period']

    # Define pre and post windows considering half-time boundaries
    pre_events = events[(events.total_seconds >= t - W_seconds) & 
                        (events.total_seconds < t) & 
                        (events.period == period)]
    
    post_events = events[(events.total_seconds > t) & 
                         (events.total_seconds <= t + W_seconds) & 
                         (events.period == period)]

    event_type = event_row.get('event_type', None)
    
    # Calculate metrics for both windows
    pre_m = get_player_metrics(pre_events, target_player_id, event_type=event_type, is_post_window=False)
    post_m = get_player_metrics(post_events, target_player_id, event_type=event_type, is_post_window=True)

    baseline_stats = get_player_match_baseline_stats(events, target_player_id, W_seconds)

    # Combine into a single row
    output = event_row.to_dict()
    for metric in pre_m.keys():
        output[f'pre_{metric}'] = pre_m[metric]
        output[f'post_{metric}'] = post_m[metric]
        output[f'delta_{metric}'] = post_m[metric] - pre_m[metric]

        # Z-score calculation based on baseline statistics for post-event metrics
        mean_key = f'mean_{metric}'
        std_key = f'std_{metric}'
        
        # Only calculate z-scores if baseline statistics are available
        if baseline_stats:
            mean_val = baseline_stats[mean_key]
            std_val = baseline_stats[std_key]
            
            # 1. option: There is a standard deviation, so we can calculate the z-score
            if std_val > 0:
                output[f'z_score_{metric}'] = (post_m[metric] - mean_val) / std_val
            
            # 2. option: There is no standard deviation (player always performs the same), and the current performance is the same
            elif post_m[metric] == mean_val:
                output[f'z_score_{metric}'] = 0.0
                
            # 3. option: There is no standard deviation, but the current performance is different (statistically meaningless z-score)
            else:
                output[f'z_score_{metric}'] = np.nan
                
        # 4. option: There is no baseline data available (e.g., the player played too little)
        else:
            output[f'z_score_{metric}'] = np.nan
    
    # Home/Away indicator
    output['is_home'] = 1 if team == home_team_name else 0

    # Compute red card difference in the game
    events_before_t = events[events.total_seconds <= t]
    focal_reds = 0
    opp_reds = 0
    
    if 'foul_committed_card' in events.columns:
        reds = events_before_t[events_before_t['foul_committed_card'].isin(['Red Card', 'Second Yellow'])]
        focal_reds += len(reds[reds.team == team])
        opp_reds += len(reds[reds.team != team])
        
    if 'bad_behaviour_card' in events.columns:
        reds_bb = events_before_t[events_before_t['bad_behaviour_card'].isin(['Red Card', 'Second Yellow'])]
        focal_reds += len(reds_bb[reds_bb.team == team])
        opp_reds += len(reds_bb[reds_bb.team != team])
        
    output['red_card_diff'] = focal_reds - opp_reds

    # y_conceded / y_score calculation based on MODE
    events_next_10m = events[(events.total_seconds > t) & (events.total_seconds <= t + 600)]
    
    target_met = 0
    if 'shot_outcome' in events_next_10m.columns:
        if MODE == 'PDE':
            # PDE: Did the team concede a goal? (The opposing team scored a goal)
            goals = events_next_10m[(events_next_10m.type == 'Shot') & 
                                    (events_next_10m.shot_outcome == 'Goal') & 
                                    (events_next_10m.team != team)]
        elif MODE == 'PFE':
            # PFE: Did the team score a goal? (The examined team scored a goal)
            goals = events_next_10m[(events_next_10m.type == 'Shot') & 
                                    (events_next_10m.shot_outcome == 'Goal') & 
                                    (events_next_10m.team == team)]
            
        if len(goals) > 0:
            target_met = 1

    output['y'] = target_met

    return output

In [6]:
# Read all match files to extract home team names
files = glob.glob("sb_offline_data/match_*.pkl")

home_teams_dict = {}
match_ids = []
missing = 0

for file in files:
    # Get match ID from filename
    m_id = int(os.path.basename(file).split('_')[1].split('.')[0])
    match_ids.append(m_id)
    
    # Load events
    events = pd.read_pickle(file)

    # Extract the home team from the Starting XI events
    starting_xis = events[events['type'] == 'Starting XI']
    if not starting_xis.empty:
        home_team = starting_xis.iloc[0]['team']
        home_teams_dict[m_id] = home_team
    else:
        home_teams_dict[m_id] = "Unknown"
        missing += 1

print(f"Processed {len(match_ids)} matches. Missing home team info for {missing} matches.")

Processed 1238 matches. Missing home team info for 0 matches.


In [7]:
final_individual_data = []

# Going through each match and its Events to calculate the behavioral response metrics
for m_id, match_events in df.groupby('match_id'):
    
    current_home_team = home_teams_dict.get(m_id, "Unknown")

    events = pd.read_pickle(f"sb_offline_data/match_{m_id}.pkl")
    events['total_seconds'] = events['minute'] * 60 + events['second']

    events['x'] = events['location'].str[0]
    events['y'] = events['location'].str[1]
    
    if 'pass_end_location' in events.columns:
        events['end_x'] = events['pass_end_location'].str[0]
        events['end_y'] = events['pass_end_location'].str[1]
    else:
        events['end_x'] = np.nan
        events['end_y'] = np.nan
    
    for _, event_row in match_events.iterrows():

        # Calculate metrics
        rich_event = calculate_individual_behavioral_response(event_row, events, current_home_team)

        final_individual_data.append(rich_event)

# Full DataFrame
master_individual_df = pd.DataFrame(final_individual_data)

# Save the final DataFrame to CSV based on MODE
if MODE == 'PDE':
    master_individual_df.to_csv('output/individual/catalogue/protagonists_master.csv', index=False, encoding='utf-8-sig')